In [ ]:
from main import*
from circle_utilities import*
from datasets import sphere

from IPython.display import clear_output # To clear tqdm bars

DTM parameters

In [ ]:
m = 0.1
p = 2
beta = 2
dtm_arg = DTM_arg(m, p, beta)

precision_segment = 20

Identify the geodesic : run the FDTM algorithm on a uniform circle grid to detect how many segments make the geodesic.

In [ ]:
N = 500

# Endpoints
theta_x, theta_y = -1/12, -11/12
x = np.array([np.cos(np.pi*theta_x), np.sin(np.pi*theta_x)])
y = np.array([np.cos(np.pi*theta_y), np.sin(np.pi*theta_y)])

# Uniform point cloud on the circle
X = circle_grid(N)

# Get indices of closest point in the grid to x and y
i, j = np.argmin(norm(X - x, axis=1)), np.argmin(norm(X - y, axis=1))

dtm = DTM(X, dtm_arg, dtm_circle)
M = FDTM(X, dtm).select_edges().make_edges(precision_segment=precision_segment, runtime=True).shortest_path(source=i)
path = M.path((i, j))[0]

clear_output()

In [ ]:
l0 = len(path)  # Amount of points on the geodesic
segments_lengths = norm(X[path[1:]] - X[path[:-1]], axis=1)
l = 1 + np.sum(segments_lengths > segments_lengths.max() * 0.8)  # Ignore small edges that would result of approximation error

if l != l0:
    print(f'Segment lengths: {segments_lengths}')
    print('Warning : not counting smaller segments')

assert abs(theta_x-theta_y) <= 1 and N > 20*l  # Arguments should be at most 1 apart and the amount of points on the grid should be large enough compare to the number of points on the geodesic
theta_geodesic = np.linspace(theta_x, theta_y, l)  # Actual intermediate points (angles) of the true geodesic
geodesic = np.stack((np.cos(np.pi*theta_geodesic), np.sin(np.pi*theta_geodesic)), axis=1)  # Truc geodesic points

Compute the FDTM of the geodesics by using only the points from the geodesics as a point cloud

In [ ]:
precision_segment_finer = 100000  # Very high precision

M = FDTM(geodesic, dtm).make_edges(precision_segment=precision_segment_finer).shortest_path().make_path((0, l-1))
true_distance = M.distances[0, l-1]
clear_output()

Convergence speed of the empirical FDTM : To be more efficient, instead of searching for the shortest path in sample points, we compute the weight of the path closest to the true geodesic we have identified.

In [ ]:
base = 2
start = 8
end = 16
k = end - start + 1
n_samples = np.logspace(start, end, num=k, base=base, dtype=int)  # Number of samples range from base**start to base**stop
print(f'Number of samples: {n_samples}')

avg = 500

In [ ]:
metrics = k*[None]
endpoints = k*[None]
error = np.zeros((k, avg))

seed = 16  # Random seed

for u in tqdm(range(k), dynamic_ncols=True):
    for v in tqdm(range(avg), leave=False, dynamic_ncols=True):
        Xn = sphere(n_samples[u], 1, seed=seed+v)
        
        dtmn = DTM(Xn, dtm_arg, compute_DTM_at_X=False)

        points = np.concatenate(([x, y], Xn[np.argmin(cdist(Xn, geodesic), axis=0)]))  # Point cloud obtained by projecting the true geodesic intemediate points onto Xn
        
        Mn = FDTM(points, dtmn).make_edges(precision_segment=precision_segment).shortest_path(0).make_path((0, 1)) # To go faster, we compute the FDTM using only the edges that approximate the true geodesic
        error[u, v] = abs(Mn.distances[0][1] - true_distance)
        
        if v==0: # Store once per n value to plot
            metrics[u] = Mn
            endpoints[u] = (0, 1)
            
clear_output()

Plot

In [ ]:
save = True  # Save figures

In [ ]:
mean, std = np.mean(error, axis=1), np.std(error, axis=1)
log_n, log_mean = np.log(n_samples), np.log(mean)
emp_exp, _ = np.polyfit(log_n, log_mean, 1)  # Empirical exponent by linear regression

fig, ax = plt.subplots()

ax.loglog(n_samples, mean, label='Average empirical offset', c='teal')
ax.fill_between(n_samples, mean - std, mean + std, alpha=0.3, label="±1 standard deviation", color='teal')

# Empirical rate of convergence
empirical_rate = n_samples**emp_exp
empirical_rate *= mean[0] / empirical_rate[0]  # Shift curve to start at same point as data
ax.loglog(n_samples, empirical_rate, linestyle='dashed', label=f'Empirical convergence rate: {emp_exp:.3f}', c='darkblue', alpha=0.6)

upp_exp = - 1/2
theoretical_rate = n_samples**upp_exp
theoretical_rate *= mean[0] / theoretical_rate[0]  # Shift curve to start at same point as data
ax.loglog(n_samples, theoretical_rate, linestyle='dashed', label=f'Theoretical convergence rate: {upp_exp}', c='darkorange')

ax.legend(loc='lower left')
ax.set_xlabel('Number of sample points')
ax.set_ylabel('Offset from true FDTM')

ax.grid(True, linestyle='--', alpha=0.6)
ax.grid(True, which='minor', linestyle=':', alpha=0.3)

if save : fig.savefig(f"figures\\circle_convergence.png", dpi=300)

ax.set_title('Convergence of the Empirical FDTM between two endpoints')

# plt.gcf().set_dpi(100)
plt.show()